# The Trading League
### EP0418 · optional side quest · opens after Lab 3 · stays open until the end of Week 7 · results at Lab 7

**Scoreboard:** https://stanleysp-eng.github.io/EP0418-labs/

**You need:** an internet connection and the League guide (a PDF linked from the scoreboard). The notebook fetches everything else.

**This is not a lab.** Nothing here is graded or required. It is a game for anyone who wants one, and it runs beside the labs.

---
**The game, in plain words.** Nothing here is real money. The scorer pretends you
put $1 into SPY, a fund that owns a slice of the 500 biggest American companies,
five years ago, and replays history to see what your rule would have turned it
into. Every evening the rule makes one choice for tomorrow: **keep the share** (1),
**hold cash** (0), or **bet it will fall** (-1). A rule is just a sentence that makes
that choice, for example *keep it while the price is above its 50-day average*.

You get the last five years of SPY prices to try rules on. The class scoreboard
shows how every rule did on those five years. Then, after entries close at the end
of Week 7, the market runs for four weeks that nobody has seen yet, and every rule
is scored on **those** weeks. The rule that looked best on the past is usually not
the one that wins. Finding out why is the point.

**Five rules of the game**

1. Your entry is a function `strategy(prices)` that gives 1, 0 or -1 for every day.
2. A choice for a day may use that day's price and earlier ones. Never later ones.
3. The scorer is locked. Each time the rule switches, the pretend pot shrinks by
   0.1%, the way a broker's fee would. A jumpy rule pays it many times.
4. The rule to beat is **buy-and-hold**: keep the share the whole time and do nothing.
5. Enter as often as you like; the latest entry counts. Nothing is graded.

Paper trading only. Prices come from Yahoo Finance through **yfinance**, a free
library for personal, educational use. Nothing here is investment advice.

---
**How the game uses what the labs teach.** Topic 2 gave you a loop: **prompt →
generate → verify → refine**, and a warning: *it ran without an error* is the
weakest evidence there is. Lab 3 gave you `shift()` and rolling averages, which
is all a trading rule is made of. Tasks 1 to 3 take about an hour and leave you
with an entry on the board. Tasks 4 to 6 are for later, as the labs hand you more
tools.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

Every prompt in these notebooks opens with the rules that matter for that task:
do not drop rows, show the row count before and after, treat -999 as missing,
explain the code in plain words. Keep those lines when you adapt a prompt, and
start your own prompts the same way.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True

from ep0418 import backtest

if not os.path.exists("spy_daily.csv"):                  # fetched once, then reused
    try:
        import yfinance as yf          # if this fails: run  !pip install yfinance  first
        h = yf.Ticker("SPY").history(period="5y", interval="1d")
        pd.DataFrame({"Date": pd.to_datetime(h.index).tz_localize(None).normalize(),
                      "Close": h["Close"].values}).to_csv("spy_daily.csv", index=False)
    except Exception as e:
        print("Yahoo did not answer:", e, "- wait a minute and run this cell again.")

prices = pd.read_csv("spy_daily.csv", parse_dates=["Date"])
print(f"SPY: {len(prices):,} trading days, {prices.Date.min().date()} to {prices.Date.max().date()}, "
      f"latest ${prices.Close.iloc[-1]:,.2f}")
show.series(prices, col="Close", title="SPY closing price, five years")

position = strategy = None     # placeholders, so later cells give a readable message

## Task 1 — Look at the data before you trust it

*Topic 2B: the three ways data arrives, and the five questions to ask of any new dataset.*

The cell above got the prices the third way in the notes: a **live API**. The
yfinance library asked Yahoo's website, got back **JSON** (boxes inside boxes),
turned it into a table, and saved the table as a **CSV** so you only fetch once.
All three forms, one cell.

Now the five questions from the notes, in order. Run the cell and read every line.

In [ ]:
print("1. Rows and columns:", prices.shape, list(prices.columns))
print("2. Period:", prices.Date.min().date(), "to", prices.Date.max().date(),
      "| largest gap between rows:", prices.Date.diff().max().days, "days")
print("3. Close min / max: ${:,.2f} / ${:,.2f}".format(prices.Close.min(), prices.Close.max()))
print("4. Blanks:", int(prices.isna().sum().sum()),
      "| rows with Close <= 0 (a sentinel pretending to be a price):", int((prices.Close <= 0).sum()))
r = prices.Close.pct_change()
print("5. Largest one-day move: {:+.1%} on {}".format(r.loc[r.abs().idxmax()], prices.Date[r.abs().idxmax()].date()))

**What you should see.** About 1,250 rows, two columns, five years. The largest gap
between rows is three or four days, not one. Why? The market shuts at weekends and
holidays, so those gaps are not missing data. Lab 3 makes that distinction exact.
No blanks, no zero prices. The biggest one-day move is bigger than most people
guess; look up the date and find out what happened that day.

**Write it down:** the strangest number on screen, and why it is there.

**Strangest number:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

The gap is the one to notice: the largest gap between rows is three or four days,
and there are hundreds of two- and three-day gaps, yet the "blanks" line says zero.
Nothing is missing. Weekends and public holidays simply have no trading day. A
reading of "no data" and "no event" look identical in a date column until you
know what produced the data. The other candidate is the largest one-day move:
far bigger than a typical day, and tied to a news event you can look up.
</details>

## Task 2 — What does doing nothing earn, and what do costs do?

*Topic 2A: "it ran without an error" is the weakest evidence. Here the evidence is a number you can argue with.*

First the rule to beat. Buy-and-hold means: buy the share on day one, never touch
it. The scorer prints four numbers. **Total return** is how much the share grew.
**Sharpe ratio** is the one the board ranks by: think of it as *return per unit of
bumpiness*. Two rules that both made 50% are not equal if one gave you a smooth
ride and the other a roller coaster; the smooth one has the higher Sharpe ratio.
Above 1 is good; above 3 on five years of a stock index is suspicious.

In [ ]:
bh = backtest.buy_and_hold(prices)
print("Buy-and-hold, five years of SPY")
print(bh)

Now a ready-made rule, so you can see one work before you write your own. A
**moving average** is the average price over the last so-many days. A **crossover**
rule compares a short one with a long one: keep the share while the average of the
last 20 days is above the average of the last 100 days (recent prices are higher
than older ones, so the price is trending up), and hold cash otherwise.

The cell runs it twice: once as if switching were free, once with the pretend pot
losing 0.1% per switch, as a real broker's fee would take it. Then it runs a jumpier
version that compares the last 5 days with the last 20. A 5-day average wiggles far
more, so the two lines cross far more often and the rule switches far more often.

In [ ]:
def crossover(prices, fast=20, slow=100):
    close = prices["Close"]
    f = close.rolling(fast).mean()
    s = close.rolling(slow).mean()
    return (f > s).astype(int).where(s.notna(), 0)     # 0 until the slow window fills

for fast, slow in [(20, 100), (5, 20)]:
    pos = crossover(prices, fast, slow)
    free = backtest.run(prices, pos, cost=0)
    paid = backtest.run(prices, pos)
    print(f"crossover {fast}/{slow}: {paid.trades} trades  "
          f"Sharpe {free.sharpe:.2f} without costs, {paid.sharpe:.2f} with costs  "
          f"(total {free.total:+.1%} -> {paid.total:+.1%})")

show.flags(prices.set_index("Date").Close, crossover(prices).values == 1,
           title="SPY, five years; marked where the 20/100 crossover holds the share", label="holding the share")

**What you should see.** The 5/20 version trades several times more often than the
20/100 one, and costs eat a much bigger slice of its return. On the chart, the
marked stretches are when the rule is holding the share; the gaps are when it sat
in cash. Ask yourself: did it sit out the falls, or the rises?

**Write it down:** how many trades each version made, and how much of the jumpy
one's return the costs ate.

**Trades and costs:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

The slow rule (20/100) switches about a dozen times in five years; the jumpy one
(5/20) switches several times more. Fees barely touch the slow one. For the jumpy
one, compare its total return without and with fees: the difference is the fees,
and it is a large slice of what it earned, so after fees the two rules end up close.

On the chart: red is "holding the share", the bare line is "in cash". The rule sat
out most of the long fall in 2022, which is its whole purpose. But look at each
rebound: it gets back in late, after the price has already recovered part of the
way, because a 100-day average takes weeks to turn. Sitting out falls and missing
the start of rises is the crossover's trade-off.
</details>

## Task 3 — Your first entry, with Gemini

*Topic 2A: the whole loop. Plan first, then prompt, generate, verify, refine.*

You are not going to invent a rule today. The crossover in Task 2 compares a
20-day average with a 100-day average. You are going to keep the 20 and change the
**100** to a number of your own, say 50, 150 or 200, with Gemini's help, and get
that rule onto the board. That is enough to walk the whole loop once.

**Plan first.** Before asking for any code, ask Gemini to think with you. Put the
number you are considering in place of 50 if you like:

> I am new to this. Answer in three plain sentences, no headings, no bullet
> points, no formulas, no code. I hold a stock when its 20-day average price is
> above its 100-day average, and hold cash otherwise. If I used 50 days instead of
> 100, would the rule switch more often or less often, and why?

If you get an essay with headings and formulas anyway, reply **"simpler, three
sentences"**. That is the refine step; assistants over-answer, and telling them
the shape you want is normal.

Read the three sentences. If they make sense, settle on the length of your long
average (the number replacing 100). Then **prompt** for the code, with the rules that matter
stated up front, putting your length where it says [your number]:

> Do not use any information from later rows. Explain in plain words what the code
> does. I have a pandas DataFrame `prices` with columns Date and Close, one row per
> trading day. Write a Python function `strategy(prices)` that returns a pandas
> Series of 1 or 0 for each day: 1 when the 20-day rolling mean of Close is above
> the [your number]-day rolling mean, else 0. Fill the rows before the long window
> is full with 0.

**Generate** is Gemini's job. **Verify** is yours: paste the function into the cell
below in place of the `TODO` body, run it, and read the three checks. **Refine**:
if a check fails, tell Gemini exactly which one and what it said, not "it didn't
work".

In [ ]:
def strategy(prices):
    # TODO(you): replace this body with the function Gemini wrote. Return 1, 0 or -1 for every row.
    position = ...
    return position


if strategy(prices.head(150)) is not ...:
    position = strategy(prices)

# ---- verification contract — do not edit ----
check.position_is_valid(position, prices)   # every day has a 1, 0 or -1
check.no_lookahead(strategy, prices)        # cuts the end off the file; your earlier choices must not change
check.trades_at_least(position, minimum=4)  # it changed its mind at least four times in five years
check.report()

**The three checks, in plain words.** The first says every day got an answer. The
third says the rule is not buy-and-hold wearing a hat. The second is the clever
one: it chops the last months off the file and runs your rule again. If any of
your *earlier* choices changed, your rule was peeking at the future. Code that
peeks looks brilliant on the past and is useless tomorrow.

Now score it against buy-and-hold.

In [ ]:
if filled(position):
    mine = backtest.run(prices, position)
    print("Your rule, five years of SPY"); print(mine)
    print("\nBuy-and-hold"); print(bh)
    check.compare_to_baseline(mine.sharpe, bh.sharpe, metric="Sharpe", lower_is_better=False)
    check.report()
    show.flags(prices.set_index("Date").Close, pd.Series(position).values == 1,
               title="SPY, five years; marked where your rule holds the share", label="holding the share")

**If you got a NOTE** saying your rule did not beat buy-and-hold: that is normal.
SPY mostly went up for five years, so doing nothing was hard to beat. It costs you
nothing here, and the hidden four weeks may tell a different story. Write down
what buy-and-hold caught that your rule missed.

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

Buy-and-hold was in the market on every rising day, including the sharp rebounds
that follow a fall. A crossover is in cash on exactly those days, because its
averages have not turned yet. Over five years in which SPY roughly doubled, those
missed rebound days are most of the gap. A rule beats buy-and-hold on the past
mainly by sitting out the falls; whether the next four weeks contain a fall is what
the hidden period decides.
</details>

Now go to the **Submit** cell at the bottom, put in your admin number, and run it.
Your entry is on the board. Everything from here on is for the coming weeks.

---
# The weeks after today

The league stays open until the end of Week 7. Each lab hands you a new tool, and
each lab's closing cell will remind you. You do not have to do any of this in one
sitting.

## Task 4 — Your own rule

*Topic 2A: a good prompt has four parts: your data, your goal, your constraint, your check.*

Pick an idea from the League guide (trend, breakout,
mean reversion, volatility filter). Ask Gemini for a plan first, as in Task 3. Then
prompt with all four parts:

> Do not use any information from later rows. Explain in plain words what it does.
> I have a pandas DataFrame `prices` with columns Date and Close, one row per
> trading day. Write a Python function `strategy(prices)` that returns a pandas
> Series of the same length holding 1, 0 or -1 for each day, decided using only
> that day's row and earlier rows. Use [describe your idea here]. Fill the rows
> before any rolling window is full with 0.

Paste it into the Task 3 cell in place of the crossover, run the checks and the
score, then run Submit again. The latest entry counts.

## Task 5 — Rehearse the hidden weeks (after Lab 6)

*Topic 2A: verify means more than "it ran". Here you check two rows by hand, except the rows are months.*

You cannot see the four hidden weeks. You can pretend. The cell pretends the file
ended six months ago: it scores your rule on the months **before** that (which it
could have been tuned on) and on the months **after** (which it could not). The
second number is the honest one. Then it does the same for last month alone.

One rule: run it **once** and write the numbers down. If you change your rule
because the second number looked bad, the second number has become part of what
you tuned on, and you are back to square one.

In [ ]:
if filled(position):
    cut = prices.Date < prices.Date.max() - pd.Timedelta(days=182)
    before = backtest.run(prices[cut], strategy(prices[cut]))
    after_pos = strategy(prices).loc[~cut]           # decided with the full history, scored on the tail
    after = backtest.run(prices[~cut], after_pos)
    bh_after = backtest.buy_and_hold(prices[~cut])
    print(f"tuned on   {cut.sum():,} days: Sharpe {before.sharpe:.2f}  total {before.total:+.1%}")
    print(f"scored on  {(~cut).sum():,} days: Sharpe {after.sharpe:.2f}  total {after.total:+.1%}"
          f"   (buy-and-hold on the same days: Sharpe {bh_after.sharpe:.2f}, total {bh_after.total:+.1%})")
    month = prices.Date >= prices.Date.max() - pd.Timedelta(days=30)
    last_month = backtest.run(prices[month], strategy(prices).loc[month]); bh_month = backtest.buy_and_hold(prices[month])
    print(f"last month {int(month.sum())} days: return {last_month.total:+.2%}   (buy-and-hold {bh_month.total:+.2%})")

**Did the second number hold up?** If the Sharpe ratio halved or changed sign, the
rule learned the past, not the market. Say so in the write-up; the write-up prize
is for honesty, not for a high number.

## Task 6 — The write-up (two answers, the second podium)

*Topic 2A: refine means saying exactly what was wrong. This is where you prove you read the code.*

Two answers, a few sentences each, in the `WRITEUP` box of the Submit cell:

1. **What your rule does and when it loses**, in words a friend outside the course
   would understand.
2. **One mistake Gemini made** while writing or changing your rule, how you caught
   it, and what you changed. Everyone has one. Weak: "it made a mistake, I fixed it."
   Strong: "its first version used a centred window; the lookahead check failed; I
   asked for a trailing window and the rehearsal Sharpe fell from 2.1 to 0.4."

---
## Submit (today, and every time you improve the rule)

Put in your admin number and run the cell. It reruns the three checks and sends
nothing if one fails. If they pass, it sends your rule, your write-up and your
five-year score to the league, and your entry appears on the scoreboard within a
minute. Entries close at the end of Week 7; the exact time is on the scoreboard.

In [ ]:
ADMIN_NO = "..."          # TODO(you): your admin number, e.g. 2512345
WRITEUP = """
TODO(you): your two write-up answers (Task 6). Leave as is for today's first entry.
"""

FORM_URL = ""
FIELDS = {'admin': '', 'code': '', 'writeup': '', 'sharpe': '', 'total': '', 'trades': '', 'rehearsal': ''}

if ADMIN_NO != "..." and filled(position):
    # the same three checks as Task 3, rerun here: nothing goes to the board unverified
    check.position_is_valid(position, prices)
    check.no_lookahead(strategy, prices)
    check.trades_at_least(position, minimum=4)
    ok = check.report(verbose=False)
if ADMIN_NO != "..." and filled(position) and not ok:
    print("Not sent. A check above failed; fix the rule in Task 3, rerun it, then run this cell again.")
if ADMIN_NO != "..." and filled(position) and ok:
    import inspect, urllib.parse
    try:
        source = inspect.getsource(strategy)
    except OSError:
        source = None
        print("Could not read the function's source. Run the cell that defines "
              "strategy() once more, then run this cell again.")
    mine = backtest.run(prices, position)
    cut = prices.Date < prices.Date.max() - pd.Timedelta(days=182)
    reh_before = backtest.run(prices[cut], strategy(prices[cut])).sharpe
    reh_after = backtest.run(prices[~cut], strategy(prices).loc[~cut]).sharpe
    entry = {"admin": ADMIN_NO, "code": source or "", "writeup": WRITEUP.strip(),
             "sharpe": f"{mine.sharpe:.3f}", "total": f"{mine.total:.4f}", "trades": str(mine.trades),
             "rehearsal": f"before {reh_before:.2f} / after {reh_after:.2f}"}
    if FORM_URL and source:
        data = urllib.parse.urlencode({FIELDS[k]: v for k, v in entry.items()}).encode()
        try:
            urllib.request.urlopen(urllib.request.Request(FORM_URL, data=data,
                                   headers={"User-Agent": "EP0418"}), timeout=60)
            print(f"Entered: {ADMIN_NO}, five-year Sharpe {mine.sharpe:.2f}, {mine.trades} trades. "
                  "Check the scoreboard in a minute.")
        except Exception as e:
            print("The league did not answer:", e, "- try again in a minute.")
    else:
        out = f"strategy_{ADMIN_NO}.py"
        TRIPLE = chr(34) * 3
        with open(out, "w") as f:
            f.write(TRIPLE + WRITEUP.strip() + chr(10) + TRIPLE + chr(10) * 2)
            f.write("import numpy as np" + chr(10) + "import pandas as pd" + chr(10) * 3)
            f.write(source or "# strategy() source missing: see the message above" + chr(10))
        print("saved", out, "- upload it to the League folder on Brightspace.")
        try:
            from google.colab import files
            files.download(out)
        except Exception:
            print("Download it from the folder icon on the left.")

---
## Read before you run

An assistant was asked for a rule that holds SPY on days when the price is rising and goes to cash otherwise. It wrote the code below, and the backtest over five years showed a Sharpe ratio above 6.

**Do not run it yet.** Read it, answer the two questions below,
and only then remove the `#` and run it to see whether you were right.

Which day's close is each row comparing with which? Could a trader at the close of day t have known it?

Write down what `check.no_lookahead` will report, and what this rule would earn on a week whose prices have not happened yet.

<details><summary>Hint 1: where to look</summary>

Which row does `shift(-1)` take its value from: the one before today, or the one after?

</details>
<details><summary>Hint 2: closer to the answer</summary>

Shifting by minus one brings tomorrow's close into today's row. The rule is long on every day that is about to go up. The lookahead check fails at the first cut; on the real week after the deadline the rule has no tomorrow to read.

</details>

**After you have written your answer, and not before:** select the cell, click the
pencil-and-spark icon at its top right and choose **Explain code**. Did Gemini notice
the fault? It often does not. That is the point: an explanation that reads well is
not evidence the code is right.

**To fix it:** choose **Transform code** on the same cell and say what to change, in
plain words. Colab shows the change as a diff before it is applied. Read the diff. If
it touches a `check.` line, reject it.

In [ ]:
# --- AI-generated. Read it first. ---
# rising = prices["Close"].shift(-1) > prices["Close"]
# position = rising.astype(int)

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Quiz yourself before you leave (5 minutes)

The checks tell you whether the code is right. They do not tell you whether
**you** know what it did. Paste this into the Gemini chat:

> Quiz me on this lab. Ask five short questions, one at a time, in plain words, on these ideas: the four steps of the loop (prompt, generate, verify, refine) and the step most people skip; why 'it ran without an error' is weak evidence, and what the three checks add; what buy-and-hold is and why it is the rule to beat; what lookahead is, in plain words, and how the cut-the-end-off check catches it; why a rule that trades often loses to costs. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks in the chat panel; you answer in the chat box; it marks you and asks
the next. Ignore the suggestion chips under its messages. Answer in your own
words, without looking back until it has marked you.

Then write your score and the idea to revise below. Your tutor asks for them in
the plenary.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

---
## Before you leave

- Your entry is on the scoreboard. Look for your admin number.
- Three numbers written down: the strangest number in the data, buy-and-hold's
  Sharpe ratio, and how much costs ate from the jumpy crossover.
- You can say in one sentence what the loop is, and which step most people skip.

Come back after Lab 5 (a volatility filter for Task 4), Lab 6 (Tasks 5 and 6), and
whenever you have a better rule. At Lab 7 the tutor shows two boards: the five years you could see
and the four weeks you could not. Look at who moved, and which way. That
reshuffle is the whole lesson of the league.

**Mistake caught:** *(what I expected · what was there · how I found it)*